In [1]:
import torch
import torchvision
from torchvision import transforms

## Step-1: Dataset Load/Creation

In [2]:
root = '../data/'

In [3]:
train_dataset = torchvision.datasets.FashionMNIST(
    root=root,
    train=True,
    download=True
)

type(train_dataset), len(train_dataset)

(torchvision.datasets.mnist.FashionMNIST, 60000)

In [4]:
train_dataset

Dataset FashionMNIST
    Number of datapoints: 60000
    Root location: ../data/
    Split: Train

In [5]:
sample = train_dataset[0]

sample

(<PIL.Image.Image image mode=L size=28x28>, 9)

> A PyTorch Dataset represents a collection of individual examples and defines how to retrieve one example by index.

In [6]:
image, label = sample

type(image), type(label)

(PIL.Image.Image, int)

> A Dataset is not a batch of tensors. Think of it primarily as an indexed collection

## Dataset Exploration

In [7]:
print("Image type:", type(image))
print("Label type:", type(label))
print("Label:", label)
print("Image size:", image.size)
print("Image mode:", image.mode)

Image type: <class 'PIL.Image.Image'>
Label type: <class 'int'>
Label: 9
Image size: (28, 28)
Image mode: L


In [8]:
# train_dataset[0:5]

> a PyTorch Dataset is indexable, but it should not automatically be thought of like a NumPy array or pandas DataFrame with fully supported slicing.

In [9]:
print(type(train_dataset.data))
print(train_dataset.data.shape)
print(train_dataset.data.dtype)

print(type(train_dataset.targets))
print(train_dataset.targets.shape)
print(train_dataset.targets.dtype)

<class 'torch.Tensor'>
torch.Size([60000, 28, 28])
torch.uint8
<class 'torch.Tensor'>
torch.Size([60000])
torch.int64


In [10]:
train_dataset.data[0].shape

torch.Size([28, 28])

> So internally MNIST already stores its raw images as a tensor yet `train_dataset[0]` returns a PIL image.
> 
> `dataset.data` is the raw internal storage, while `dataset[i]` goes through the dataset's retrieval logic and returns the sample in the public format defined by `MNIST.__getitem__`.

## Transform with `ToTensor()`

In [11]:
train_dataset_tensor = torchvision.datasets.FashionMNIST(
    root=root,
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

type(train_dataset_tensor), len(train_dataset_tensor)

(torchvision.datasets.mnist.FashionMNIST, 60000)

In [12]:
sample = train_dataset_tensor[0]

image, label = sample

print(type(image))
print(type(label))
print(image.shape)
print(image.dtype)
print(image.min(), image.max())

<class 'torch.Tensor'>
<class 'int'>
torch.Size([1, 28, 28])
torch.float32
tensor(0.) tensor(1.)


In [13]:
print(train_dataset_tensor.data[0].dtype)
print(train_dataset_tensor.data[0].min(), train_dataset_tensor.data[0].max())

image, _ = train_dataset_tensor[0]

print(image.dtype)
print(image.min(), image.max())

torch.uint8
tensor(0, dtype=torch.uint8) tensor(255, dtype=torch.uint8)
torch.float32
tensor(0.) tensor(1.)


In [14]:
raw_image = train_dataset_tensor.data[0]
transformed_image, _ = train_dataset_tensor[0]

print(raw_image.shape)
print(transformed_image.shape)

torch.Size([28, 28])
torch.Size([1, 28, 28])


> The transform does not modify `dataset.data` itself. It is applied when the sample is retrieved.
>
> `ToTensor()`:
> - changes integer pixel value (0-255) to floating point value (0.0-1.0)
> - adds channel: 1 - grayscale, 3 - rgb

## Chain Transforms using Compose

In [21]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset_norm = torchvision.datasets.FashionMNIST(
    root=root,
    train=True,
    download=True,
    transform=transform
)

In [22]:
img_tensor, _ = train_dataset_tensor[0]
img_norm, _ = train_dataset_norm[0]

print("Before normalization:")
print(img_tensor.min(), img_tensor.max())

print("After normalization:")
print(img_norm.min(), img_norm.max())

Before normalization:
tensor(0.) tensor(1.)
After normalization:
tensor(-0.4242) tensor(2.8215)


## Subsets and Train/Val Splits

In [23]:
len(train_dataset_tensor)

60000

In [29]:
g = torch.Generator().manual_seed(42)

train_set, val_set = torch.utils.data.random_split(
    train_dataset_tensor,
    [50000, 10000],
    generator=g
)

len(train_set), len(val_set)

(50000, 10000)

In [34]:
type(train_set)

torch.utils.data.dataset.Subset

In [30]:
train_set.indices[:5], val_set.indices[:5]

([37542, 43087, 2992, 25810, 39634], [59753, 9414, 10988, 15501, 49861])

> `random_split()` did not copy 50,000 images into a new dataset.
>
> instead, created a subset storing 50,000 indices for the train set

In [32]:
train_set.dataset is train_dataset_tensor

True

> the subset also stores reference to the underlying dataset

## DataLoader

#### Why DataLoader exists
> A Dataset answers:
> - "How do I retrieve one example?"

> A DataLoader answers:
> - "How do I efficiently iterate through that dataset in batches?"

In [35]:
train_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=64,
    shuffle=True
)

type(train_loader)

torch.utils.data.dataloader.DataLoader

In [36]:
len(train_set), len(train_loader)

(50000, 782)

In [39]:
images, labels = next(iter(train_loader))

images.shape, labels.shape

(torch.Size([64, 1, 28, 28]), torch.Size([64]))